<a href="https://colab.research.google.com/github/Stepa555/ecommerce-dashboard/blob/main/%D0%90%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85_%D1%8D%D0%BB%D0%B5%D0%BA%D1%82%D1%80%D0%BE%D0%BD%D0%BD%D0%BE%D0%B9_%D0%BA%D0%BE%D0%BC%D0%BC%D0%B5%D1%80%D1%86%D0%B8%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving order_transaction.csv to order_transaction.csv
Saving user_behavior.csv to user_behavior.csv


In [ ]:
!pip install pandasql

import pandas as pd
import numpy as np
from pandasql import sqldf
import matplotlib.pyplot as plt
import seaborn as sns

pysqldf = lambda q: sqldf(q, globals())


  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=2ca412fec3395ff2dcf1f070f0ee74a6bc891212922f3261b49425cd779df0f4
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


In [ ]:
user = pd.read_csv('user_behavior.csv')
orders = pd.read_csv('order_transaction.csv')


# Разведка данных


In [ ]:
print('=== USER BEHAVIOR ===')
print(f'Размер: {user.shape}')
print(f'Колонки: {list(user.columns)}')
print(f'Пропуски:\n{user.isnull().sum()}')

print("\n=== ORDER TRANSACTION ===")
print(f"Размер: {orders.shape}")
print(f"Колонки: {list(orders.columns)}")
print(f"Пропуски:\n{orders.isnull().sum()}")

# Посмотрим примеры
print("\nПример user:")
print(user.head(2))

print("\nПример orders:")
print(orders.head(2))

=== USER BEHAVIOR ===
Размер: (3150, 17)
Колонки: ['user_id', 'behavior_id', 'behavior_type', 'behavior_time', 'behavior_duration', 'device_type', 'app_version', 'user_age', 'user_gender', 'user_city', 'user_level', 'last_login_time', 'page_view_count', 'click_count', 'collect_count', 'cart_count', 'network_type']
Пропуски:
user_id                0
behavior_id            0
behavior_type          0
behavior_time          0
behavior_duration     89
device_type            0
app_version            0
user_age             185
user_gender            0
user_city              0
user_level             0
last_login_time      176
page_view_count        0
click_count            0
collect_count         99
cart_count           101
network_type           0
dtype: int64

=== ORDER TRANSACTION ===
Размер: (2575, 13)
Колонки: ['order_id', 'user_id', 'order_time', 'payment_time', 'payment_amount', 'payment_method', 'product_category', 'product_count', 'discount_amount', 'shipping_fee', 'logistics_company'

In [ ]:
user.info()
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3150 entries, 0 to 3149
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            3150 non-null   object 
 1   behavior_id        3150 non-null   object 
 2   behavior_type      3150 non-null   object 
 3   behavior_time      3150 non-null   object 
 4   behavior_duration  3061 non-null   float64
 5   device_type        3150 non-null   object 
 6   app_version        3150 non-null   object 
 7   user_age           2965 non-null   float64
 8   user_gender        3150 non-null   object 
 9   user_city          3150 non-null   object 
 10  user_level         3150 non-null   object 
 11  last_login_time    2974 non-null   object 
 12  page_view_count    3150 non-null   int64  
 13  click_count        3150 non-null   int64  
 14  collect_count      3051 non-null   float64
 15  cart_count         3049 non-null   float64
 16  network_type       3150 

# Очистка данных

In [ ]:
# Копируем, чтобы не испортить оригинал

user = user.copy()
orders = orders.copy()

# 1. Даты

user['behavior_time'] = pd.to_datetime(user['behavior_time'], errors='coerce')
user['last_login_time'] = pd.to_datetime(user['last_login_time'], errors='coerce')
orders['order_time'] = pd.to_datetime(orders['order_time'], errors='coerce')
orders['payment_time'] = pd.to_datetime(orders['payment_time'], errors='coerce')

# 2. Числовые колонки

user['user_age'] = pd.to_numeric(user['user_age'], errors='coerce')
user['behavior_duration'] = pd.to_numeric(user['behavior_duration'], errors='coerce')
orders['discount_amount'] = pd.to_numeric(orders['discount_amount'], errors='coerce')
orders['logistics_time'] = pd.to_numeric(orders['logistics_time'], errors='coerce')

# 3. Заполнение пропусков

user['user_age'].fillna(user['user_age'].median(), inplace=True)
user['last_login_time'].fillna(user['behavior_time'].min(), inplace=True)
user['behavior_duration'].fillna(user['behavior_duration'].median(), inplace=True)
user['collect_count'].fillna(0, inplace=True)
user['cart_count'].fillna(0, inplace=True)


orders['discount_amount'].fillna(0, inplace=True)
orders['logistics_time'].fillna(orders['logistics_time'].median(), inplace=True)

print("✅ Очистка завершена")
print(f"Осталось пропусков в user: {user.isnull().sum().sum()}")
print(f"Осталось пропусков в orders: {orders.isnull().sum().sum()}")

✅ Очистка завершена
Осталось пропусков в user: 0
Осталось пропусков в orders: 116


/tmp/ipykernel_889/3607050648.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  user['user_age'].fillna(user['user_age'].median(), inplace=True)
/tmp/ipykernel_889/3607050648.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inpl

# SQL-запросы

In [ ]:
# Запрос 1: JOIN таблиц

query1 = """
SELECT u.user_id, u.user_gender, u.user_city, u.user_level,
       o.order_id, o.payment_amount, o.product_category
FROM user u
LEFT JOIN orders o ON u.user_id = o.user_id
LIMIT 10
"""
print("✅ JOIN:")
print(pysqldf(query1))

# Запрос 2: GROUP BY — выручка по категориям

query2 = """
SELECT product_category,
       COUNT(*) as orders_count,
       SUM(payment_amount) as revenue,
       AVG(payment_amount) as avg_check
FROM orders
GROUP BY product_category
ORDER BY revenue DESC
"""
print("\n✅ Топ категорий по выручке:")
print(pysqldf(query2))

# Запрос 3: оконная функция — первые заказы пользователей

query3 = """
SELECT user_id, order_id, payment_amount,
       ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY order_time) as order_num
FROM orders
WHERE payment_amount IS NOT NULL
LIMIT 15
"""
print("\n✅ Оконная функция ROW_NUMBER:")
print(pysqldf(query3))

# Запрос 4: CASE WHEN — сегментация по возрасту
query4 = """
SELECT user_id, user_age,
       CASE WHEN user_age < 25 THEN 'Young'
            WHEN user_age BETWEEN 25 AND 40 THEN 'Adult'
            ELSE 'Senior'
       END as age_group
FROM user
LIMIT 10
"""
print("\n✅ CASE WHEN (сегментация):")
print(pysqldf(query4))

✅ JOIN:
  user_id user_gender user_city user_level order_id  payment_amount  \
0    U382     Unknown     Wuhan        VIP    O0342         2217.33   
1    U382     Unknown     Wuhan        VIP    O0479         1570.36   
2    U382     Unknown     Wuhan        VIP    O2106         1751.96   
3    U382     Unknown     Wuhan        VIP    O2459         1988.87   
4    U117     Unknown     Wuhan        New    O0580         1694.67   
5    U117     Unknown     Wuhan        New    O0943         2393.92   
6    U117     Unknown     Wuhan        New    O1493         1880.26   
7    U138      Female  Hangzhou        New    O0106         2130.49   
8    U138      Female  Hangzhou        New    O0259         1776.35   
9    U138      Female  Hangzhou        New    O0796         2086.28   

  product_category  
0       Appliances  
1       Appliances  
2       Home_Goods  
3          Apparel  
4       Home_Goods  
5       Appliances  
6             Food  
7          Apparel  
8             Food  


# Анализ и метрики

In [ ]:
# Конверсия

users_with_orders = orders['user_id'].nunique()
total_user = user['user_id'].nunique()
conversion = users_with_orders / total_user * 100

# Выручка

total_revenue = orders['payment_amount'].sum()
avg_order = orders['payment_amount'].mean()


# Скидки
total_discount = orders['discount_amount'].sum()
avg_discount = orders[orders['discount_amount'] > 0]['discount_amount'].mean()

# Возвраты
refund_rate = (orders['refund_reason'] != 'No_Refund').mean() * 100

print("="*50)
print("📊 КЛЮЧЕВЫЕ МЕТРИКИ")
print("="*50)
print(f"Конверсия в покупку: {conversion:.1f}%")
print(f"Общая выручка: {total_revenue:,.0f}")
print(f"Средний чек: {avg_order:.0f}")
print(f"Общая сумма скидок: {total_discount:,.0f}")
print(f"Средняя скидка (по заказам со скидкой): {avg_discount:.0f}")
print(f"Доля возвратов: {refund_rate:.1f}%")

📊 КЛЮЧЕВЫЕ МЕТРИКИ
Конверсия в покупку: 99.6%
Общая выручка: 3,197,518
Средний чек: 1242
Общая сумма скидок: 544,857
Средняя скидка (по заказам со скидкой): 219
Доля возвратов: 8.8%


In [ ]:
# Сохраняем чистые данные в CSV
user.to_csv('user_clean.csv', index=False)
orders.to_csv('orders_clean.csv', index=False)

# Скачиваем на компьютер
from google.colab import files
files.download('user_clean.csv')
files.download('orders_clean.csv')

print("✅ Файлы скачаны. Теперь можно открыть их в Power BI")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Файлы скачаны. Теперь можно открыть их в Power BI
